# 💳 Credit Card Fraud Detection — ML & DL Capstone
**Dataset:** [Credit Card Cheating Detection (CCCD), Kaggle](https://www.kaggle.com/datasets/arslanali4343/credit-card-cheating-detection-cccd) · **Task:** binary classification (fraud vs. genuine)

| Stage | What is done |
|---|---|
| 1. Setup & data | install libs, download data (kagglehub / upload / Drive) |
| 2. EDA | class imbalance, amount & time behaviour, feature–target correlation |
| 3. Preprocessing | cleaning, stratified train / validation / test split, scaling |
| 4. ML models | Logistic Regression (SMOTE), Random Forest (SMOTE), XGBoost (cost-sensitive) |
| 5. DL models | Deep MLP (class-weighted), Autoencoder (unsupervised anomaly detection) |
| 6. Evaluation | PR-AUC, ROC-AUC, Precision, Recall, F1, MCC, confusion matrices, threshold tuning |
| 7. Deployment | save artifacts → Streamlit app → public URL from Colab |

**Methodological notes**
* Fraud is rare (well under 1 %), so **accuracy is misleading**; the primary metric is **PR-AUC (Average Precision)**, with Recall / F1 / MCC as secondary.
* Resampling (SMOTE) is applied **only to the training split** to avoid leakage into validation/test.
* Decision thresholds are **tuned on the validation set** (max F1) and reported on the untouched **test set**.

## 1. Setup

In [ ]:
!pip -q install imbalanced-learn xgboost kagglehub streamlit plotly
import os, glob, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix, classification_report,
                             precision_recall_curve, roc_curve)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_style("whitegrid")
ART = "artifacts"; os.makedirs(ART, exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Load the dataset
Three options — the first one that works is used:
1. `LOCAL_CSV` — path to the CSV (e.g. on Google Drive after mounting).
2. **kagglehub** download (public dataset; if it asks for credentials, upload `kaggle.json` or use option 3).
3. Manual upload of the CSV from your computer.

In [ ]:
LOCAL_CSV = ""   # e.g. "/content/drive/MyDrive/creditcard.csv"

def load_data():
    if LOCAL_CSV and os.path.exists(LOCAL_CSV):
        return pd.read_csv(LOCAL_CSV), LOCAL_CSV
    try:
        import kagglehub
        path = kagglehub.dataset_download("arslanali4343/credit-card-cheating-detection-cccd")
        csvs = sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True), key=os.path.getsize)
        if csvs:
            return pd.read_csv(csvs[-1]), csvs[-1]
    except Exception as e:
        print("kagglehub download failed:", e)
    from google.colab import files
    up = files.upload()
    name = list(up.keys())[0]
    return pd.read_csv(name), name

df, SRC = load_data()
print("Loaded:", SRC, "| shape:", df.shape)
df.head()

In [ ]:
# Identify the target column automatically (Class / is_fraud / label ...), else the last column
cands = [c for c in df.columns if c.strip().lower() in ("class", "is_fraud", "isfraud", "fraud", "label", "target")]
TARGET = cands[0] if cands else df.columns[-1]
print("Target column:", TARGET)
df.info()
display(df.describe().T.head(12))
print("\nMissing values per column (non-zero):\n", df.isna().sum()[df.isna().sum() > 0])

## 3. Exploratory Data Analysis

In [ ]:
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").round().astype(int)
counts = df[TARGET].value_counts().sort_index()
print(counts, f"\nFraud rate: {counts.get(1,0)/len(df)*100:.4f}%  (imbalance ≈ 1:{counts[0]//max(counts.get(1,1),1)})")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=TARGET, data=df, ax=ax[0], palette=["#4c72b0", "#dd5555"])
ax[0].set_title("Class distribution"); ax[0].set_yscale("log"); ax[0].set_xticklabels(["Genuine", "Fraud"])
ax[1].pie(counts, labels=["Genuine", "Fraud"], autopct="%1.3f%%", colors=["#4c72b0", "#dd5555"], startangle=90)
ax[1].set_title("Class share"); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
if "Amount" in df.columns:
    sns.boxplot(x=TARGET, y="Amount", data=df, ax=ax[0], showfliers=False, palette=["#4c72b0", "#dd5555"])
    ax[0].set_title("Transaction amount by class (no outliers)")
if "Time" in df.columns:
    for lbl, col in [(0, "#4c72b0"), (1, "#dd5555")]:
        sns.kdeplot((df.loc[df[TARGET] == lbl, "Time"] / 3600) % 24, ax=ax[1], label=["Genuine", "Fraud"][lbl],
                    fill=True, color=col, common_norm=False)
    ax[1].set_title("Hour-of-day density by class"); ax[1].set_xlabel("hour"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
num = df.select_dtypes(include=np.number)
corr = num.corr()[TARGET].drop(TARGET).sort_values()
plt.figure(figsize=(14, 4))
corr.plot(kind="bar", color=np.where(corr > 0, "#dd5555", "#4c72b0"))
plt.title(f"Pearson correlation of each feature with {TARGET}"); plt.tight_layout(); plt.show()

top = corr.abs().sort_values(ascending=False).head(6).index
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for a, f in zip(axes.ravel(), top):
    sns.kdeplot(data=df, x=f, hue=TARGET, common_norm=False, fill=True, ax=a, palette=["#4c72b0", "#dd5555"])
    a.set_title(f)
plt.suptitle("Most discriminative features: class-conditional densities"); plt.tight_layout(); plt.show()

## 4. Preprocessing
* Drop exact duplicate rows; keep numeric features; impute missing values with the **training** median.
* Stratified split **70 % train / 15 % validation / 15 % test**.
* `StandardScaler` fitted on the training split only (PCA features V1–V28 are already centred; this mainly rescales `Time` and `Amount`).

In [ ]:
before = len(df); df = df.drop_duplicates(); print(f"Removed {before-len(df)} duplicate rows")
non_num = [c for c in df.columns if c != TARGET and not pd.api.types.is_numeric_dtype(df[c])]
if non_num: print("Dropping non-numeric columns:", non_num)
FEATURES = [c for c in df.columns if c != TARGET and c not in non_num]

X = df[FEATURES].astype("float64"); y = df[TARGET].values
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

medians = X_tr.median()
X_tr, X_va, X_te = X_tr.fillna(medians), X_va.fillna(medians), X_te.fillna(medians)
scaler = StandardScaler().fit(X_tr.values)
Xtr, Xva, Xte = [scaler.transform(a.values).astype("float32") for a in (X_tr, X_va, X_te)]
for n, yy in [("train", y_tr), ("val", y_va), ("test", y_te)]:
    print(f"{n:5s}: {len(yy):7d} rows, {yy.sum():4d} frauds ({yy.mean()*100:.3f}%)")

In [ ]:
# SMOTE on the TRAINING split only (used by LR and RF)
t = time.time()
Xtr_sm, ytr_sm = SMOTE(sampling_strategy=0.1, random_state=SEED).fit_resample(Xtr, y_tr)
print(f"After SMOTE: {len(ytr_sm)} rows, fraud share {ytr_sm.mean()*100:.1f}%  ({time.time()-t:.1f}s)")

## 5. Evaluation helpers
`evaluate()` picks the threshold that maximises F1 on the **validation** set, then reports test metrics at that threshold.

In [ ]:
results, curves, confusions, thresholds, val_scores, test_scores = {}, {}, {}, {}, {}, {}

def best_threshold(y_true, s):
    p, r, t = precision_recall_curve(y_true, s)
    f1 = 2 * p * r / (p + r + 1e-12)
    i = np.nanargmax(f1[:-1]); return float(t[i])

def evaluate(name, s_val, s_test):
    thr = best_threshold(y_va, s_val)
    pred = (s_test >= thr).astype(int)
    m = {"PR_AUC": average_precision_score(y_te, s_test), "ROC_AUC": roc_auc_score(y_te, s_test),
         "Precision": precision_score(y_te, pred, zero_division=0), "Recall": recall_score(y_te, pred),
         "F1": f1_score(y_te, pred), "MCC": matthews_corrcoef(y_te, pred)}
    results[name] = {k: round(float(v), 4) for k, v in m.items()}
    thresholds[name] = thr; val_scores[name] = s_val; test_scores[name] = s_test
    confusions[name] = confusion_matrix(y_te, pred, labels=[0, 1]).tolist()
    fpr, tpr, _ = roc_curve(y_te, s_test); pr, rc, _ = precision_recall_curve(y_te, s_test)
    idx = lambda a: np.linspace(0, len(a) - 1, min(len(a), 300)).astype(int)   # downsample for the app
    curves[name] = {"fpr": fpr[idx(fpr)].tolist(), "tpr": tpr[idx(tpr)].tolist(),
                    "prec": pr[idx(pr)].tolist(), "rec": rc[idx(rc)].tolist()}
    print(f"== {name} ==  threshold={thr:.4f}");  print(pd.Series(results[name]).to_string())
    print(classification_report(y_te, pred, target_names=["Genuine", "Fraud"], digits=4))
    plt.figure(figsize=(3.6, 3)); sns.heatmap(confusions[name], annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=["Genuine", "Fraud"], yticklabels=["Genuine", "Fraud"])
    plt.title(name); plt.ylabel("Actual"); plt.xlabel("Predicted"); plt.tight_layout(); plt.show()

## 6. Machine-learning models
### 6.1 Logistic Regression (baseline, trained on SMOTE data)

In [ ]:
t = time.time()
lr = LogisticRegression(max_iter=2000, C=0.1, random_state=SEED).fit(Xtr_sm, ytr_sm)
print(f"trained in {time.time()-t:.1f}s")
evaluate("Logistic Regression", lr.predict_proba(Xva)[:, 1], lr.predict_proba(Xte)[:, 1])

### 6.2 Random Forest (SMOTE data)

In [ ]:
t = time.time()
rf = RandomForestClassifier(n_estimators=150, min_samples_leaf=2, max_features="sqrt",
                            n_jobs=-1, random_state=SEED).fit(Xtr_sm, ytr_sm)
print(f"trained in {time.time()-t:.1f}s")
evaluate("Random Forest", rf.predict_proba(Xva)[:, 1], rf.predict_proba(Xte)[:, 1])

### 6.3 XGBoost (cost-sensitive via `scale_pos_weight`, original imbalanced data, early stopping on validation)

In [ ]:
spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
t = time.time()
xgb = XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    scale_pos_weight=spw, eval_metric="aucpr", early_stopping_rounds=50,
                    tree_method="hist", random_state=SEED, n_jobs=-1)
xgb.fit(Xtr, y_tr, eval_set=[(Xva, y_va)], verbose=False)
print(f"trained in {time.time()-t:.1f}s | best iteration {xgb.best_iteration} | scale_pos_weight={spw:.1f}")
evaluate("XGBoost", xgb.predict_proba(Xva)[:, 1], xgb.predict_proba(Xte)[:, 1])

imp = pd.Series(xgb.get_booster().get_score(importance_type="gain"))
imp.index = [FEATURES[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k for k in imp.index]
imp = imp.sort_values(ascending=False)
imp.head(15).sort_values().plot(kind="barh", figsize=(7, 5), title="XGBoost feature importance (gain)"); plt.show()

## 7. Deep-learning models
### 7.1 Deep MLP — supervised, class-weighted binary cross-entropy
Architecture: 64 → 32 → 16 (ReLU, BatchNorm, Dropout) → sigmoid. Early stopping on validation PR-AUC.

In [ ]:
neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
class_weight = {0: 1.0, 1: min(neg / pos, 50.0)}   # capped to keep training stable

def build_mlp(d):
    m = keras.Sequential([layers.Input((d,)),
        layers.Dense(64, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(32, activation="relu"), layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(16, activation="relu"), layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"), keras.metrics.Recall(name="recall")])
    return m

mlp = build_mlp(Xtr.shape[1]); mlp.summary()
es = keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=6, restore_best_weights=True)
hist = mlp.fit(Xtr, y_tr, validation_data=(Xva, y_va), epochs=40, batch_size=2048,
               class_weight=class_weight, callbacks=[es], verbose=2)
pd.DataFrame(hist.history)[["loss", "val_loss", "pr_auc", "val_pr_auc"]].plot(figsize=(8, 3.5), title="MLP training"); plt.show()
evaluate("Deep MLP", mlp.predict(Xva, verbose=0).ravel(), mlp.predict(Xte, verbose=0).ravel())

### 7.2 Autoencoder — unsupervised anomaly detection
Trained **only on genuine transactions** to reconstruct them. Frauds reconstruct poorly, so the **reconstruction MSE** is used as the anomaly score (no labels needed during training — useful when labelled fraud is scarce).

In [ ]:
Xtr_norm = Xtr[y_tr == 0]
d = Xtr.shape[1]
inp = layers.Input((d,))
h = layers.Dense(20, activation="elu")(inp); h = layers.Dense(10, activation="elu")(h)
z = layers.Dense(6, activation="elu", name="bottleneck")(h)
h = layers.Dense(10, activation="elu")(z); h = layers.Dense(20, activation="elu")(h)
out = layers.Dense(d, activation="linear")(h)
ae = keras.Model(inp, out); ae.compile(optimizer="adam", loss="mse"); ae.summary()
Xva_norm = Xva[y_va == 0]
hist = ae.fit(Xtr_norm, Xtr_norm, validation_data=(Xva_norm, Xva_norm), epochs=30, batch_size=1024,
              callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)], verbose=2)

recon_err = lambda X: np.mean(np.square(X - ae.predict(X, verbose=0)), axis=1)
e_va, e_te = recon_err(Xva), recon_err(Xte)
plt.figure(figsize=(8, 3.5))
plt.hist(e_te[y_te == 0], bins=100, alpha=.6, density=True, label="Genuine", log=True)
plt.hist(e_te[y_te == 1], bins=100, alpha=.6, density=True, label="Fraud", log=True)
plt.xlim(0, np.percentile(e_te, 99.9)); plt.legend(); plt.title("Reconstruction error (test)"); plt.show()
evaluate("Autoencoder", e_va, e_te)

## 8. Model comparison

In [ ]:
res = pd.DataFrame(results).T.sort_values("PR_AUC", ascending=False)
display(res.style.background_gradient(cmap="Greens", axis=0).format("{:.4f}"))
BEST = res.index[0]; print("Best model by test PR-AUC:", BEST)

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
res[["PR_AUC", "ROC_AUC", "F1", "Recall", "Precision", "MCC"]].plot(kind="bar", ax=ax[0], rot=20)
ax[0].set_title("Test metrics"); ax[0].set_ylim(0, 1.05); ax[0].legend(fontsize=8)
for n, c in curves.items():
    ax[1].plot(c["fpr"], c["tpr"], label=f"{n} ({results[n]['ROC_AUC']:.3f})")
    ax[2].plot(c["rec"], c["prec"], label=f"{n} ({results[n]['PR_AUC']:.3f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=.8); ax[1].set(title="ROC", xlabel="FPR", ylabel="TPR"); ax[1].legend(fontsize=8)
ax[2].set(title="Precision–Recall", xlabel="Recall", ylabel="Precision"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Interpretation guide for the report**
* Compare **PR-AUC** first — ROC-AUC is usually > 0.95 for all models on this data and hides differences under heavy imbalance.
* Tree ensembles (XGBoost / RF) typically lead; the MLP is competitive; the Autoencoder scores lower but needs **no fraud labels**.
* Business trade-off: lowering the threshold raises Recall (fewer missed frauds) at the cost of Precision (more false alarms). The Streamlit app lets you explore this interactively.

## 9. Save artifacts for the Streamlit app

In [ ]:
joblib.dump(scaler, f"{ART}/scaler.joblib")
joblib.dump(lr, f"{ART}/logreg.joblib"); joblib.dump(rf, f"{ART}/rf.joblib", compress=3); joblib.dump(xgb, f"{ART}/xgb.joblib")
mlp.save(f"{ART}/mlp.keras"); ae.save(f"{ART}/autoencoder.keras")

models_meta = {
    "Logistic Regression": {"file": "logreg.joblib", "kind": "sklearn", "type": "ml"},
    "Random Forest":       {"file": "rf.joblib", "kind": "sklearn", "type": "ml"},
    "XGBoost":             {"file": "xgb.joblib", "kind": "sklearn", "type": "ml"},
    "Deep MLP":            {"file": "mlp.keras", "kind": "keras", "type": "mlp"},
    "Autoencoder":         {"file": "autoencoder.keras", "kind": "keras", "type": "autoencoder"}}
for n in models_meta: models_meta[n]["threshold"] = thresholds[n]

meta = {"dataset": "Credit Card Cheating Detection (CCCD)", "target": TARGET, "features": FEATURES,
        "medians": {k: float(v) for k, v in medians.items()}, "n_rows": int(len(df)),
        "fraud_rate": float(df[TARGET].mean()), "models": models_meta, "best_model": BEST,
        "test_metrics": results, "confusion": confusions, "curves": curves,
        "feature_importance": {k: float(v) for k, v in imp.items()}}
json.dump(meta, open(f"{ART}/meta.json", "w"))

# test sample for the app: all test frauds + 2000 genuine
te = X_te.copy(); te[TARGET] = y_te
sample = pd.concat([te[te[TARGET] == 1], te[te[TARGET] == 0].sample(min(2000, (te[TARGET] == 0).sum()), random_state=SEED)])
sample.sample(frac=1, random_state=SEED).reset_index(drop=True).to_csv(f"{ART}/test_sample.csv", index=False)
!ls -lh artifacts

## 10. Streamlit app
The cell below writes `app.py` (single-transaction prediction, batch CSV scoring, and a performance dashboard).

In [ ]:
%%writefile app.py
"""
Credit Card Fraud Detection – Streamlit App
Loads the models trained in the Colab notebook (artifacts/ folder) and serves:
  1. Single-transaction prediction (pick a test transaction or enter values)
  2. Batch prediction on an uploaded CSV (with downloadable results)
  3. Model performance dashboard (metrics, confusion matrices, ROC / PR curves)
"""
import json
import os

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

ART = os.environ.get("ARTIFACT_DIR", "artifacts")

st.set_page_config(page_title="Credit Card Fraud Detection", page_icon="💳", layout="wide")


# ----------------------------------------------------------------------------- loading
@st.cache_resource
def load_artifacts():
    with open(os.path.join(ART, "meta.json")) as f:
        meta = json.load(f)
    scaler = joblib.load(os.path.join(ART, "scaler.joblib"))
    models = {}
    for name, info in meta["models"].items():
        path = os.path.join(ART, info["file"])
        if info["kind"] == "sklearn":
            models[name] = joblib.load(path)
        else:  # keras
            from tensorflow import keras
            models[name] = keras.models.load_model(path, compile=False)
    return meta, scaler, models


@st.cache_data
def load_sample():
    p = os.path.join(ART, "test_sample.csv")
    return pd.read_csv(p) if os.path.exists(p) else None


def score(model_name, X_scaled, meta, models):
    """Return a fraud score in [0, 1]-ish for each row (AE returns reconstruction error)."""
    info = meta["models"][model_name]
    m = models[model_name]
    if info["kind"] == "sklearn":
        return m.predict_proba(X_scaled)[:, 1]
    if info["type"] == "autoencoder":
        recon = m.predict(X_scaled, verbose=0)
        return np.mean(np.square(X_scaled - recon), axis=1)
    return m.predict(X_scaled, verbose=0).ravel()


def prepare(df, meta, scaler):
    feats = meta["features"]
    missing = [c for c in feats if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    X = df[feats].apply(pd.to_numeric, errors="coerce")
    X = X.fillna(pd.Series(meta["medians"]))
    return scaler.transform(X.values).astype("float32")


# ----------------------------------------------------------------------------- UI
try:
    meta, scaler, models = load_artifacts()
except Exception as e:
    st.error(f"Could not load artifacts from '{ART}/'. Run the Colab notebook first.\n\n{e}")
    st.stop()

sample = load_sample()
model_names = list(meta["models"].keys())
best = meta.get("best_model", model_names[0])

st.sidebar.title("💳 Fraud Detector")
model_name = st.sidebar.selectbox("Model", model_names, index=model_names.index(best))
info = meta["models"][model_name]
is_ae = info["type"] == "autoencoder"
default_thr = float(info["threshold"])
if is_ae:
    thr = st.sidebar.number_input("Anomaly threshold (reconstruction MSE)",
                                  value=default_thr, format="%.5f")
else:
    thr = st.sidebar.slider("Decision threshold", 0.0, 1.0, default_thr, 0.01)
st.sidebar.caption(f"Tuned threshold (max F1 on validation): **{default_thr:.4f}**")
st.sidebar.caption(f"Best model on test PR-AUC: **{best}**")

st.title("Credit Card Fraud Detection")
st.caption(f"Dataset: {meta['dataset']} · {meta['n_rows']:,} transactions · "
           f"fraud rate {meta['fraud_rate']*100:.3f}%")

tab1, tab2, tab3 = st.tabs(["🔍 Single transaction", "📂 Batch prediction", "📊 Model performance"])

# ---- Tab 1: single transaction
with tab1:
    feats = meta["features"]
    mode = st.radio("Input", ["Pick from test set", "Enter manually"], horizontal=True)
    if mode == "Pick from test set" and sample is not None:
        c1, c2 = st.columns([1, 2])
        with c1:
            which = st.selectbox("Show", ["Any", "Fraud only", "Genuine only"])
            pool = sample
            if which == "Fraud only":
                pool = sample[sample[meta["target"]] == 1]
            elif which == "Genuine only":
                pool = sample[sample[meta["target"]] == 0]
            if st.button("🎲 Random transaction") or "row_idx" not in st.session_state \
                    or st.session_state.get("pool") != which:
                st.session_state.row_idx = int(np.random.choice(pool.index))
                st.session_state.pool = which
        row = sample.loc[[st.session_state.row_idx]]
        with c2:
            st.dataframe(row, width="stretch")
        true_label = int(row[meta["target"]].iloc[0])
    else:
        vals = {}
        cols = st.columns(5)
        for i, f in enumerate(feats):
            vals[f] = cols[i % 5].number_input(f, value=float(meta["medians"][f]), format="%.4f")
        row = pd.DataFrame([vals])
        true_label = None

    Xs = prepare(row, meta, scaler)
    s = float(score(model_name, Xs, meta, models)[0])
    pred = int(s >= thr)

    c1, c2, c3 = st.columns(3)
    c1.metric("Prediction", "🚨 FRAUD" if pred else "✅ Genuine")
    c2.metric("Anomaly score" if is_ae else "Fraud probability", f"{s:.4f}")
    if true_label is not None:
        c3.metric("Actual label", "Fraud" if true_label else "Genuine",
                  "correct" if true_label == pred else "wrong",
                  delta_color="normal" if true_label == pred else "inverse")

    if not is_ae:
        fig = go.Figure(go.Indicator(mode="gauge+number", value=s * 100,
                                     number={"suffix": "%"},
                                     gauge={"axis": {"range": [0, 100]},
                                            "bar": {"color": "crimson" if pred else "seagreen"},
                                            "threshold": {"line": {"color": "black", "width": 3},
                                                          "value": thr * 100}}))
        fig.update_layout(height=260, margin=dict(t=20, b=10))
        st.plotly_chart(fig, width="stretch")

    st.subheader("All models on this transaction")
    rows = []
    for n in model_names:
        sc = float(score(n, Xs, meta, models)[0])
        t = float(meta["models"][n]["threshold"])
        rows.append({"Model": n, "Score": round(sc, 5), "Threshold": round(t, 5),
                     "Prediction": "Fraud" if sc >= t else "Genuine"})
    st.dataframe(pd.DataFrame(rows), width="stretch", hide_index=True)

# ---- Tab 2: batch
with tab2:
    st.write(f"Upload a CSV with these columns: `{', '.join(meta['features'])}` "
             f"(a `{meta['target']}` column is optional – if present, metrics are computed).")
    up = st.file_uploader("CSV file", type="csv")
    if up is None and sample is not None and st.checkbox("Use bundled test sample instead"):
        up_df = sample.copy()
    elif up is not None:
        up_df = pd.read_csv(up)
    else:
        up_df = None

    if up_df is not None:
        try:
            Xs = prepare(up_df, meta, scaler)
            sc = score(model_name, Xs, meta, models)
            out = up_df.copy()
            out["fraud_score"] = sc
            out["prediction"] = (sc >= thr).astype(int)
            c1, c2 = st.columns(2)
            c1.metric("Transactions", f"{len(out):,}")
            c2.metric("Flagged as fraud", f"{out['prediction'].sum():,}")
            if meta["target"] in out.columns:
                from sklearn.metrics import (confusion_matrix, f1_score,
                                             precision_score, recall_score)
                y = out[meta["target"]].astype(int)
                p = out["prediction"]
                m1, m2, m3 = st.columns(3)
                m1.metric("Precision", f"{precision_score(y, p, zero_division=0):.3f}")
                m2.metric("Recall", f"{recall_score(y, p, zero_division=0):.3f}")
                m3.metric("F1", f"{f1_score(y, p, zero_division=0):.3f}")
                cm = confusion_matrix(y, p, labels=[0, 1])
                st.plotly_chart(px.imshow(cm, text_auto=True, x=["Genuine", "Fraud"],
                                          y=["Genuine", "Fraud"], labels=dict(x="Predicted", y="Actual"),
                                          color_continuous_scale="Blues", height=350),
                                width="stretch")
            st.dataframe(out.sort_values("fraud_score", ascending=False).head(200),
                         width="stretch")
            st.download_button("⬇️ Download predictions", out.to_csv(index=False),
                               "fraud_predictions.csv", "text/csv")
        except Exception as e:
            st.error(str(e))

# ---- Tab 3: performance
with tab3:
    mdf = pd.DataFrame(meta["test_metrics"]).T.reset_index().rename(columns={"index": "Model"})
    st.subheader("Test-set metrics (threshold tuned on validation set)")
    st.dataframe(mdf.style.highlight_max(subset=[c for c in mdf.columns if c != "Model"],
                                         color="#c6efce"),
                 width="stretch", hide_index=True)
    st.plotly_chart(px.bar(mdf.melt(id_vars="Model", value_vars=["PR_AUC", "ROC_AUC", "F1", "Recall", "Precision"]),
                           x="variable", y="value", color="Model", barmode="group",
                           labels={"variable": "Metric", "value": "Score"}, height=400),
                    width="stretch")

    c1, c2 = st.columns(2)
    curves = meta.get("curves", {})
    if curves:
        roc = go.Figure()
        pr = go.Figure()
        for n, cv in curves.items():
            roc.add_trace(go.Scatter(x=cv["fpr"], y=cv["tpr"], name=n, mode="lines"))
            pr.add_trace(go.Scatter(x=cv["rec"], y=cv["prec"], name=n, mode="lines"))
        roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(dash="dash", color="grey"), showlegend=False))
        roc.update_layout(title="ROC curve", xaxis_title="FPR", yaxis_title="TPR", height=420)
        pr.update_layout(title="Precision–Recall curve", xaxis_title="Recall", yaxis_title="Precision", height=420)
        c1.plotly_chart(roc, width="stretch")
        c2.plotly_chart(pr, width="stretch")

    st.subheader(f"Confusion matrix – {model_name}")
    cm = np.array(meta["confusion"][model_name])
    st.plotly_chart(px.imshow(cm, text_auto=True, x=["Genuine", "Fraud"], y=["Genuine", "Fraud"],
                              labels=dict(x="Predicted", y="Actual"), color_continuous_scale="Blues",
                              height=380), width="stretch")

    if "feature_importance" in meta:
        fi = pd.Series(meta["feature_importance"]).sort_values().tail(15)
        st.subheader("Top 15 features (XGBoost gain)")
        st.plotly_chart(px.bar(fi, orientation="h", height=450, labels={"value": "Importance", "index": ""}),
                        width="stretch")

## 11. Launch the app from Colab
Starts Streamlit in the background and exposes it through a free **Cloudflare quick tunnel** — click the printed `trycloudflare.com` link. Keep this cell running while you use the app; interrupt it (■) to stop.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        print("\n🚀 Streamlit app is live at:", m.group(), "\n"); break
try:
    tunnel.wait()
except KeyboardInterrupt:
    tunnel.terminate(); print("Tunnel stopped.")

## 12. (Optional) Download everything for submission / Streamlit Community Cloud
The zip contains `app.py`, `requirements.txt` and `artifacts/`. Push these to a GitHub repo and deploy at share.streamlit.io for a permanent URL.

In [ ]:
open("requirements.txt", "w").write("streamlit\npandas\nnumpy\nscikit-learn\nxgboost\ntensorflow-cpu\njoblib\nplotly\n")
!zip -qr fraud_app.zip app.py requirements.txt artifacts
from google.colab import files; files.download("fraud_app.zip")